In [ ]:
import polars as pl
import polars.selectors as cs
import seaborn as sns
import matplotlib.pyplot as plt
from oauthlib.common import unquote

from src.features import build_features, ARCHIVELIST
from src.predict_one import predict

#global settings
pl.Config.set_engine_affinity("streaming")
pl.Config.set_tbl_rows(-1)
%load_ext autoreload
%autoreload 2

In [ ]:
df = pl.read_parquet("../data/parquets/sold_listings_20260830.parquet").lazy()
print(df.schema)
print(df.head())

In [ ]:
#Null percentage count by column
nulls = (df.select(pl.all().null_count() / pl.len() * 100)
         .unpivot(variable_name="column_name", value_name="null_percentage")
         .filter(pl.col("null_percentage") > 0)
         .sort("null_percentage", descending=True))
print(nulls)


In [ ]:
#Checks how many listings have original_price == sold_price. original_price is derived from price_drops list so
og_is_sold = df.select(pl.col('original_price'), pl.col('sold_price')).filter(pl.col("sold_price") == pl.col('original_price')).count()
print(og_is_sold)


Restrict the eval window to 2025-08 onward where coverage is at least stable-ish. May be useful later, not in v1.

In [ ]:
#First listing w styles was 1stl sold_at 2025
monthly = (df
    .group_by(pl.col('sold_at').dt.truncate('1mo').alias('month'))
    .agg(
        pl.len().alias('total'),
        (pl.col('styles') != '').sum().alias('has_styles'),
    )
    .with_columns((pl.col('has_styles') / pl.col('total')).alias('coverage'))
    .sort('month')
)
with pl.Config(tbl_rows=-1):
    print(monthly)

Restrict the eval window to 2025-06 -- 08 onward where coverage is at least stable-ish. May be useful later, not in v1.

In [ ]:
monthly = (df
    .group_by(pl.col('sold_at').dt.truncate('1mo').alias('month'))
    .agg(
        pl.len().alias('total'),
        (pl.col('country_of_origin') != 'null').sum().alias('has_country'),
    )
    .with_columns((pl.col('has_country') / pl.col('total')).alias('coverage'))
    .sort('month')
)
with pl.Config(tbl_rows=-1):
    print(monthly)

#Metrics for particular interest groups of designers - archive etc.

How many listings with the same title are there?

In [ ]:
import numpy as np

not_unique_height = df.filter(~pl.col('title').is_unique()).collect().height
titles = df.select(pl.col('title')).group_by(pl.col('title')).agg(pl.col('title').count().alias('count_title'))

result = df.join(titles, on=["title"], how="left")
counts = result.select('count_title').collect().to_numpy()


In [ ]:
from src.features import ICARELIST
from src.train import train_w_folds, TEST_END, TRAIN_END, report
from dateutil.relativedelta import relativedelta
from datetime import datetime, UTC

df = pl.read_parquet("../data/parquets/sold_listings_20260901.parquet").lazy()
def train_eval(df: pl.LazyDataFrame):
    df = build_features(df)
    df = df.with_columns(
        pl.col('primary_designer').is_in(ARCHIVELIST).alias('is_archive'),
        pl.col('primary_designer').is_in(ICARELIST).alias('brands_icare')
    )
    start = datetime(2026, 2, 1, tzinfo=UTC)
    start = start - relativedelta(months=12)
    end = datetime(2026, 3, 1, tzinfo=UTC)
    segment = 'brands_icare' #'brands_icare'
    return train_w_folds(df, start, end)
folds = train_eval(df)

In [ ]:
all = report(folds)
print(all)
print('')
onsame = report(folds, 'same_titles')
print(onsame)
print('')
unique = report(folds, 'unique_titles')
print(unique)
print('')
unseen = report(folds, 'unseen_titles')
print(unseen)
print('')
archive = report(folds, 'archivelist')
print(archive)

In [ ]:
print(df.select(pl.col('cover_photo_url').n_unique()).collect())
print(df.select(pl.len()).collect())

In [1]:
from src.predict_one import predict_one
predict_one([])

Schema({'title': String, 'department': String, 'source': String, 'category': String, 'category_path': String, 'category_size': String, 'color': String, 'condition': String, 'size': String, 'styles': String, 'country_of_origin': String, 'price': Int32, 'sold_price': Int32, 'created_at': Datetime(time_unit='us', time_zone='UTC'), 'sold_at': Datetime(time_unit='us', time_zone='UTC'), 'cover_photo_url': String, 'location': String, 'seller_id': Int32, 'seller_total_transactions': Int32, 'seller_trusted': Boolean, 'seller_rating_average': Float64, 'seller_rating_count': Int32, 'followers_count': Int32, 'heat_score': Float64, 'photo_count': Int32, 'measurement_count': Int32, 'external_id': Int64, 'currency': String, 'local_image_path': String, 'image_download_status': String, 'original_price': Int32, 'scraped_at': Datetime(time_unit='us', time_zone='UTC'), 'designer_ids': List(Int32), 'designer_names': List(String), 'id': Int64, 'photo_key': String})


0